In [1]:
from pyspark.sql import SparkSession
from pyspark.ml import Pipeline
from pyspark.sql import functions as F
import time
import psutil
from pyspark.sql import SparkSession
from pyspark.ml import Pipeline
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.feature import VectorAssembler, StringIndexer
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.mllib.evaluation import MulticlassMetrics
import numpy as np

In [2]:
# Check data hadoop
PATH_TO_CSV = "../../hdfs/data-input/drug200.parquet"
pass_linux = 'echo 123 | sudo -S'

In [3]:
# Insert data in HDFS

In [4]:
!{pass_linux} docker exec namenode hdfs dfs -mkdir -p /user/data/drug
!{pass_linux} docker cp "{PATH_TO_CSV}" namenode:/tmp/drug200.parquet
!{pass_linux} docker exec namenode hdfs dfs -put /tmp/drug200.parquet /user/data/drug
!{pass_linux} docker exec namenode hdfs dfs -ls -R /user/data/drug

[sudo] password for lenovo: [sudo] password for lenovo: Successfully copied 395MB to namenode:/tmp/drug200.parquet
[sudo] password for lenovo: put: `/user/data/drug/drug200.parquet': File exists
[sudo] password for lenovo: -rw-r--r--   2 root supergroup  395495382 2026-05-06 12:15 /user/data/drug/drug200.parquet


In [5]:
memory_use = "20g"
spark = (SparkSession.builder
    .appName("Optimized_RF_50M_Rows")
    # Allocate 12GB for drivers, leaving 4GB for the OS and Docker overhead.
    .config("spark.driver.memory", memory_use)
    .config("spark.executor.memory", memory_use)
    
    # Memory Optimization: Allocate 80% of RAM to Spark,
    # Prioritizing Execution (RF computation) over Storage (cache)
    .config("spark.memory.fraction", "0.8")
    .config("spark.memory.storageFraction", "0.2")
    
    # Data partitioning: 50 million rows should be divided into at least 500 partitions.
    .config("spark.sql.shuffle.partitions", "500")
    .config("spark.default.parallelism", "500")
    
    # Avoid timeouts when processing large volumes.
    .config("spark.network.timeout", "1200s")
    .config("spark.sql.broadcastTimeout", "1200s")

    .config("spark.jars.packages", "com.redislabs:spark-redis_2.12:3.1.0")

    .getOrCreate())

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/10 11:12:30 WARN Utils: Your hostname, lenovo-Legion-5-15IAH7H, resolves to a loopback address: 127.0.1.1; using 192.168.1.8 instead (on interface wlp0s20f3)
26/05/10 11:12:30 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/media/lenovo/D1/Data%20legion%202026/NTT_Exercises/Parallelcomputing_Distributedsystems/source-code/.venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/lenovo/.ivy2.5.2/cache
The jars for the packages stored in: /home/lenovo/.ivy2.5.2/jars
com.redislabs#spark-redis_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-0be56240-714b-49ad-9a61-93022b8c3188;1.0
	confs: [default]
	found com.redislabs#spark-redis_2.12;3.1.0 in central
	found redis.clients#jedis;3.9.0 in central
	found org.slf4j#slf4j-

In [6]:
# Read from HDFS
ip_namenode = '172.20.0.2'
spark.sparkContext.setCheckpointDir(f"hdfs://{ip_namenode}:9000/user/checkpoints")
hdfs_path = f"hdfs://{ip_namenode}:9000/user/data/drug/drug200.parquet"
df = spark.read.parquet(hdfs_path, header=True, inferSchema=True)
# Check load data
df = df.repartition(500) 
df.show(5)
df.printSchema()

+------+---+---+------+-----------+-------+-----+
|row_id|Age|Sex|    BP|Cholesterol|Na_to_K| Drug|
+------+---+---+------+-----------+-------+-----+
|698434| 35|  F|   LOW|       HIGH| 21.095|DrugY|
|749267| 18|  M|   LOW|       HIGH|  30.23|DrugY|
|479356| 53|  F|   LOW|     NORMAL| 15.839|DrugY|
|745917| 57|  M|  HIGH|       HIGH|  9.448|DrugB|
|641846| 44|  F|NORMAL|     NORMAL| 27.084|DrugY|
+------+---+---+------+-----------+-------+-----+
only showing top 5 rows
root
 |-- row_id: long (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Sex: string (nullable = true)
 |-- BP: string (nullable = true)
 |-- Cholesterol: string (nullable = true)
 |-- Na_to_K: float (nullable = true)
 |-- Drug: string (nullable = true)



In [7]:
df.count()

50000000

In [8]:
# Calculate Hardware
def get_system_usage():
    cpu_percent = psutil.cpu_percent(interval=None)
    ram_usage = psutil.virtual_memory().percent
    return cpu_percent, ram_usage

start_time = time.time()
cpu_start, ram_start = get_system_usage()

total_rows = df.count()

end_time = time.time()
cpu_end, ram_end = get_system_usage()

In [9]:
# Size data
total_cols = len(df.columns)

stats = df.select(
    F.mean("Age").alias("Average_Age"),
    F.percentile_approx("Na_to_K", 0.5).alias("Median_Na_to_K")
).collect()[0]

In [10]:
# Result calculate data
print(f"Kích thước: {total_rows} hàng x {total_cols} cột")
print(f"Trung bình Age: {stats['Average_Age']:.2f}")
print(f"Trung vị chỉ số (Na_to_K): {stats['Median_Na_to_K']}")

# Performance CPU RAM
print(f"Thời gian truy xuất (Latency): {end_time - start_time:.2f} giây")
print(f"Mức độ CPU sử dụng: {cpu_end}%")
print(f"Mức độ RAM sử dụng: {ram_end}%")

# Number of partitions
num_partitions = df.rdd.getNumPartitions()
print(f"Số lượng partitions của DataFrame: {num_partitions}")

# Distribution
partition_counts = df.withColumn("partition_id", F.spark_partition_id()) \
    .groupBy("partition_id") \
    .count()

partition_counts.show()

Kích thước: 50000000 hàng x 7 cột
Trung bình Age: 44.50
Trung vị chỉ số (Na_to_K): 20.496999740600586
Thời gian truy xuất (Latency): 2.40 giây
Mức độ CPU sử dụng: 86.1%
Mức độ RAM sử dụng: 55.6%


Số lượng partitions của DataFrame: 500


+------------+------+
|partition_id| count|
+------------+------+
|           0|100004|
|           1|100004|
|           2|100004|
|           3|100004|
|           4|100003|
|           5|100003|
|           6|100002|
|           7|100002|
|           8|100002|
|           9|100002|
|          10|100002|
|          11|100002|
|          12|100002|
|          13|100001|
|          14|100001|
|          15|100001|
|          16|100001|
|          17|100001|
|          18|100001|
|          19|100001|
+------------+------+
only showing top 20 rows


In [11]:
# Data preprocessing
# Feature Engineering
# Indexing Label
start_time_train_eval = time.time()

labelIndexer = StringIndexer(inputCol="Drug", outputCol="indexedLabel", handleInvalid="skip")

# Indexing to classification fields
sexIndexer = StringIndexer(inputCol="Sex", outputCol="Sex_Index")
bpIndexer = StringIndexer(inputCol="BP", outputCol="BP_Index")
cholIndexer = StringIndexer(inputCol="Cholesterol", outputCol="Cholesterol_Index")

# gom các đặc trưng vào một vector
featureCols = ["Age", "Sex_Index", "BP_Index", "Cholesterol_Index", "Na_to_K"]
assembler = VectorAssembler(inputCols=featureCols, outputCol="features")

rf = RandomForestClassifier(
    labelCol="indexedLabel", 
    featuresCol="features", 
    numTrees=80,           
    maxDepth=12,           
    maxBins=64,          
    subsamplingRate=0.7,  
    seed=42  
)

# Pipeline
pipeline = Pipeline(stages=[labelIndexer, sexIndexer, bpIndexer, cholIndexer, assembler, rf])

# Split Train/Test
(trainingData, testData) = df.randomSplit([0.8, 0.2], seed=42)

df.unpersist()

# Training
model = pipeline.fit(trainingData)

# Evaluate
predictions = model.transform(testData)

# Calculate Confusion Matrix by RDD API
predictionAndLabels = predictions \
.select("prediction", "indexedLabel") \
                                 .rdd.map(lambda row: (float(row.prediction),\
                                                        float(row.indexedLabel)))

evaluator = MulticlassClassificationEvaluator(labelCol="indexedLabel", predictionCol="prediction")

# Accuracy
accuracy = evaluator.evaluate(predictions, {evaluator.metricName: "accuracy"})

# Weighted F1-Score (Trong Spark f1 chính là weighted F1)
f1 = evaluator.evaluate(predictions, {evaluator.metricName: "f1"})

# Weighted Precision
weighted_precision = evaluator.evaluate(predictions, {evaluator.metricName: "weightedPrecision"})

# Weighted Recall
weighted_recall = evaluator.evaluate(predictions, {evaluator.metricName: "weightedRecall"})

print(f"Accuracy: {accuracy:.4f}")
print(f"F1-Score: {f1:.4f}")
print(f"Precision: {weighted_precision:.4f}")
print(f"Recall: {weighted_recall:.4f}")

metrics = MulticlassMetrics(predictionAndLabels)
print("Confusion Matrix:")
cm= metrics.confusionMatrix().toArray()
print(cm)

26/05/10 11:17:37 WARN MemoryStore: Not enough space to cache rdd_153_462 in memory! (computed 3.8 MiB so far)
26/05/10 11:17:37 WARN BlockManager: Persisting block rdd_153_462 to disk instead.
26/05/10 11:17:38 WARN MemoryStore: Not enough space to cache rdd_153_460 in memory! (computed 19.5 MiB so far)
26/05/10 11:17:38 WARN BlockManager: Persisting block rdd_153_460 to disk instead.
26/05/10 11:17:39 WARN MemoryStore: Not enough space to cache rdd_153_461 in memory! (computed 30.7 MiB so far)
26/05/10 11:17:39 WARN BlockManager: Persisting block rdd_153_461 to disk instead.
26/05/10 11:17:40 WARN MemoryStore: Not enough space to cache rdd_153_460 in memory! (computed 1027.4 KiB so far)
26/05/10 11:17:42 WARN MemoryStore: Not enough space to cache rdd_153_461 in memory! (computed 19.5 MiB so far)
26/05/10 11:17:43 WARN MemoryStore: Not enough space to cache rdd_153_462 in memory! (computed 30.7 MiB so far)
26/05/10 11:17:43 WARN MemoryStore: Not enough space to cache rdd_153_472 in m

Accuracy: 0.9993
F1-Score: 0.9993
Precision: 0.9993
Recall: 0.9993


/media/lenovo/D1/Data legion 2026/NTT_Exercises/Parallelcomputing_Distributedsystems/source-code/.venv/lib/python3.12/site-packages/pyspark/sql/context.py:157: FutureWarning: Deprecated in 3.0.0. Use SparkSession.builder.getOrCreate() instead.
  warnings.warn(


Confusion Matrix:


[[6.897384e+06 0.000000e+00 0.000000e+00 0.000000e+00 0.000000e+00]
 [2.335000e+03 1.031973e+06 0.000000e+00 0.000000e+00 0.000000e+00]
 [2.408000e+03 0.000000e+00 1.030856e+06 0.000000e+00 0.000000e+00]
 [1.329000e+03 0.000000e+00 0.000000e+00 6.189810e+05 0.000000e+00]
 [9.420000e+02 0.000000e+00 0.000000e+00 0.000000e+00 4.133990e+05]]


In [12]:
# from xgboost.spark import SparkXGBClassifier


# # Thay vì dùng RandomForestClassifier của Spark, ta dùng SparkXGBClassifier
# # Lưu ý: 'device'='cuda' yêu cầu Spark Cluster của bạn phải có GPU và đã cài đặt đúng Driver
# rf_gpu = SparkXGBClassifier(
#     features_col="features",
#     label_col="indexedLabel",
#     num_workers=1,              # Số lượng worker có GPU
#     device="cuda",              # QUAN TRỌNG: Ép sử dụng GPU
#     tree_method="hist",
#     num_parallel_tree=100,      # Biến XGBoost thành kiến trúc Forest
#     subsample=0.8,
#     colsample_bynode=0.8,
#     learning_rate=1.0,          # Trong SparkXGB, 'eta' thường được gọi là learning_rate
#     max_depth=12,
#     seed=42
# )

# labelIndexer = StringIndexer(inputCol="Drug", outputCol="indexedLabel", handleInvalid="skip")

# # Indexing to classification fields
# sexIndexer = StringIndexer(inputCol="Sex", outputCol="Sex_Index")
# bpIndexer = StringIndexer(inputCol="BP", outputCol="BP_Index")
# cholIndexer = StringIndexer(inputCol="Cholesterol", outputCol="Cholesterol_Index")

# # gom các đặc trưng vào một vector
# featureCols = ["Age", "Sex_Index", "BP_Index", "Cholesterol_Index", "Na_to_K"]
# assembler = VectorAssembler(inputCols=featureCols, outputCol="features")

# pipeline = Pipeline(stages=[
#     labelIndexer, 
#     sexIndexer, 
#     bpIndexer, 
#     cholIndexer, 
#     assembler, 
#     rf_gpu  
# ])

# # Chia dữ liệu
# (trainingData, testData) = df.randomSplit([0.8, 0.2], seed=42)

# # Training trên GPU
# start_time_train_eval = time.time()
# model = pipeline.fit(trainingData)
# train_time = time.time() - start_time_train_eval

# # Dự đoán
# predictions = model.transform(testData)

# # Đánh giá (Giữ nguyên phần tính toán của bạn)
# evaluator = MulticlassClassificationEvaluator(labelCol="indexedLabel", predictionCol="prediction")

# accuracy = evaluator.evaluate(predictions, {evaluator.metricName: "accuracy"})
# f1 = evaluator.evaluate(predictions, {evaluator.metricName: "f1"})

# print(f"Thời gian huấn luyện: {train_time:.2f} giây")
# print(f"Accuracy: {accuracy:.4f}")
# print(f"F1-Score: {f1:.4f}")

# # Confusion Matrix
# predictionAndLabels = predictions.select("prediction", "indexedLabel") \
#                                  .rdd.map(lambda row: (float(row.prediction), float(row.indexedLabel)))
# metrics = MulticlassMetrics(predictionAndLabels)
# print("Confusion Matrix:")
# print(metrics.confusionMatrix().toArray())

In [13]:
# Save model
model.write().overwrite().save("model_random_forest_spark")
end_time_train_eval = time.time()

In [14]:
print(f"Thời gian huấn luyện và đánh giá mô hình \
      : {end_time_train_eval - start_time_train_eval:.2f} giây")

Thời gian huấn luyện và đánh giá mô hình       : 1728.74 giây


In [15]:
import gc

In [16]:
# Free RAM
predictions.unpersist()
testData.unpersist()
trainingData.unpersist()
del predictions
del metrics
gc.collect()
spark.catalog.clearCache()

In [ ]:
# from IPython.display import display, HTML
# display(HTML("<script>Jupyter.notebook.kernel.restart()</script>"))

: 

In [ ]:
# Stop
spark.stop()
# !{pass_linux} shutdown now
import os
os._exit(0)